# Paper-to-Project: Phase 1–5 Multi-Paper Integration Test

This notebook runs the **complete backend pipeline** over **all 29 research papers** and saves a consolidated report.

| Phase | Day(s) | Description |
|-------|--------|-------------|
| Phase 1 | Day 1–4 | PDF Extraction & Multi-Engine Routing |
| Phase 2 | Day 5–8 | Canonical PaperDocument Merge & Section Audit |
| Phase 3 | Day 9–13 | Confidence, Provenance & Ingestion Benchmarks |
| Phase 4 | Day 14–18 | Semantic Chunking, Embeddings & RAG Retrieval |
| Phase 5 | Day 19–22 | Ingestion, Decomposition, Parameters, Gap Finding |

> **Estimated time:** ~3–6 min per paper. 29 papers ≈ 90–180 min total.

## Cell 0: Environment Setup — Discover All PDFs

In [2]:
import os
import sys
import json
import time
import datetime
import traceback
from collections import Counter

# ---- PATH SETUP ----
NOTEBOOK_DIR = os.path.abspath('')
if os.path.basename(NOTEBOOK_DIR) == 'tests':
    BACKEND_DIR = os.path.dirname(NOTEBOOK_DIR)
else:
    BACKEND_DIR = NOTEBOOK_DIR

if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)

PAPERS_DIR  = os.path.join(BACKEND_DIR, 'papers', 'research_papers')
REPORTS_DIR = os.path.join(BACKEND_DIR, 'tests', 'reports')
os.makedirs(REPORTS_DIR, exist_ok=True)

# ---- DISCOVER ALL PDFs ----
all_pdfs = sorted(
    [os.path.join(PAPERS_DIR, f) for f in os.listdir(PAPERS_DIR) if f.lower().endswith('.pdf')],
    key=lambda p: int(os.path.basename(p).replace('[', '').replace('].pdf', ''))
)

print(f'[ENV] Backend dir : {BACKEND_DIR}')
print(f'[ENV] Papers dir  : {PAPERS_DIR}')
print(f'[ENV] Reports dir : {REPORTS_DIR}')
print(f'[ENV] PDFs found  : {len(all_pdfs)}')
print()
for i, p in enumerate(all_pdfs, 1):
    print(f'  [{i:>2}] {os.path.basename(p)}')

[ENV] Backend dir : c:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\Projects\agentic_projects\Paper-2-Project\backend
[ENV] Papers dir  : c:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\Projects\agentic_projects\Paper-2-Project\backend\papers\research_papers
[ENV] Reports dir : c:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\Projects\agentic_projects\Paper-2-Project\backend\tests\reports
[ENV] PDFs found  : 29

  [ 1] [1].pdf
  [ 2] [2].pdf
  [ 3] [3].pdf
  [ 4] [4].pdf
  [ 5] [5].pdf
  [ 6] [6].pdf
  [ 7] [7].pdf
  [ 8] [8].pdf
  [ 9] [9].pdf
  [10] [10].pdf
  [11] [11].pdf
  [12] [12].pdf
  [13] [13].pdf
  [14] [14].pdf
  [15] [15].pdf
  [16] [16].pdf
  [17] [17].pdf
  [18] [18].pdf
  [19] [19].pdf
  [20] [20].pdf
  [21] [21].pdf
  [22] [22].pdf
  [23] [23].pdf
  [24] [24].pdf
  [25] [25].pdf
  [26] [26].pdf
  [27] [27].pdf
  [28] [28].pdf
  [29] [29].pdf


## Cell 1: Auto-Detect System Configuration

All values are **auto-detected** from your machine:
- **Model** → queried from the local Ollama REST API
- **GPU / VRAM** → detected via `torch.cuda` then `nvidia-smi` fallback
- **System RAM** → detected via `psutil`
- **Timeline** → only value you need to set manually

In [3]:
import subprocess
import requests

# ================================================================
# AUTO-DETECT: Ollama Model
# Queries the local Ollama REST API and picks the best available.
# ================================================================
PREFERRED_MODELS = [
    'qwen2.5-coder:1.5b', 'qwen2.5-coder:7b',
    'llama3', 'mistral', 'gemma'
]
MODEL_NAME = None
try:
    resp = requests.get('http://localhost:11434/api/tags', timeout=5)
    available_models = [m['name'] for m in resp.json().get('models', [])]
    for pref in PREFERRED_MODELS:
        if pref in available_models:
            MODEL_NAME = pref
            break
    if not MODEL_NAME and available_models:
        MODEL_NAME = available_models[0]
    print(f'[AUTO] Ollama models available : {available_models}')
    print(f'[AUTO] Selected model          : {MODEL_NAME}')
except Exception as e:
    MODEL_NAME = 'qwen2.5-coder:1.5b'
    print(f'[WARN] Ollama API unreachable ({e}). Fallback: {MODEL_NAME}')

# ================================================================
# AUTO-DETECT: System RAM via psutil
# ================================================================
try:
    import psutil
    system_ram_gb = round(psutil.virtual_memory().total / (1024 ** 3), 1)
    print(f'[AUTO] System RAM              : {system_ram_gb} GB')
except ImportError:
    system_ram_gb = 16.0
    print(f'[WARN] psutil not installed. Defaulting RAM = {system_ram_gb} GB')

# ================================================================
# AUTO-DETECT: GPU Name + VRAM
# Tries torch.cuda first, then nvidia-smi as fallback.
# ================================================================
gpu_model = 'CPU (no GPU detected)'
vram_gb   = 0.0
try:
    import torch
    if torch.cuda.is_available():
        gpu_model = torch.cuda.get_device_name(0)
        vram_gb   = round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 1)
        print(f'[AUTO] GPU  (torch)            : {gpu_model}')
        print(f'[AUTO] VRAM (torch)            : {vram_gb} GB')
    else:
        raise RuntimeError('CUDA not available in torch')
except Exception as torch_err:
    print(f'[WARN] torch CUDA failed ({torch_err}). Trying nvidia-smi...')
    try:
        smi_name = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            timeout=5, text=True
        ).strip().splitlines()[0]
        smi_mem = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
            timeout=5, text=True
        ).strip().splitlines()[0]
        gpu_model = smi_name
        vram_gb   = round(int(smi_mem) / 1024, 1)   # MiB -> GiB
        print(f'[AUTO] GPU  (nvidia-smi)       : {gpu_model}')
        print(f'[AUTO] VRAM (nvidia-smi)       : {vram_gb} GB')
    except Exception as smi_err:
        print(f'[WARN] nvidia-smi failed ({smi_err}). GPU constraints = 0.')

# ================================================================
# MANUAL: Only timeline needs human input
# ================================================================
TIMELINE_WEEKS = 2   # <-- change this if needed

CONSTRAINTS = {
    'gpu_model'      : gpu_model,
    'system_ram_gb'  : system_ram_gb,
    'vram_gb'        : vram_gb,
    'timeline_weeks' : TIMELINE_WEEKS
}

# ================================================================
# SCOPE: Set PAPER_LIMIT to None for all 29, or an integer
# (e.g. 3) for a quick test before the full run.
# ================================================================
PAPER_LIMIT   = None   # None = all papers
SKIP_ON_ERROR = True   # True = log errors and continue

papers_to_run = all_pdfs[:PAPER_LIMIT] if PAPER_LIMIT else all_pdfs

print()
print('--- Final Configuration ---')
print(f'  Model          : {MODEL_NAME}')
print(f'  GPU            : {gpu_model}')
print(f'  VRAM           : {vram_gb} GB')
print(f'  System RAM     : {system_ram_gb} GB')
print(f'  Timeline       : {TIMELINE_WEEKS} weeks')
print(f'  Papers to run  : {len(papers_to_run)}')
print(f'  Skip on error  : {SKIP_ON_ERROR}')

[AUTO] Ollama models available : ['nomic-embed-text:latest', 'qwen2.5-coder:1.5b']
[AUTO] Selected model          : qwen2.5-coder:1.5b
[AUTO] System RAM              : 23.6 GB
[WARN] torch CUDA failed (CUDA not available in torch). Trying nvidia-smi...
[AUTO] GPU  (nvidia-smi)       : NVIDIA GeForce RTX 5050 Laptop GPU
[AUTO] VRAM (nvidia-smi)       : 8.0 GB

--- Final Configuration ---
  Model          : qwen2.5-coder:1.5b
  GPU            : NVIDIA GeForce RTX 5050 Laptop GPU
  VRAM           : 8.0 GB
  System RAM     : 23.6 GB
  Timeline       : 2 weeks
  Papers to run  : 29
  Skip on error  : True


## Cell 2: Import Pipeline Orchestrator

In [4]:
from pipeline import graph as orchestrator
print('[OK] Pipeline orchestrator imported successfully.')

[OK] Pipeline orchestrator imported successfully.


## Cell 3: Run Pipeline Over All Papers

> Processes every paper sequentially with the full Phase 1–5 treatment.  
> Results are collected into `all_results` and failures into `all_errors`.

In [4]:
all_results = []
all_errors  = []

run_start_time = time.time()
RUN_TS = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

for paper_idx, pdf_path in enumerate(papers_to_run, 1):
    pdf_name = os.path.basename(pdf_path)
    paper_id = f'paper_{pdf_name.replace("[", "").replace("].pdf", "")}'

    print()
    print('=' * 65)
    print(f'[{paper_idx:>2}/{len(papers_to_run)}] Processing: {pdf_name}')
    print('=' * 65)

    initial_state = {
        'pdf_path'   : pdf_path,
        'constraints': CONSTRAINTS,
        'model_name' : MODEL_NAME,
        'loop_count' : 0
    }

    t0 = time.time()
    try:
        result  = orchestrator.invoke(initial_state)
        elapsed = round(time.time() - t0, 2)

        gap_rpt   = result.get('gap_report')
        ext_params = result.get('extracted_parameters')
        gap_counts = Counter(g.classification for g in gap_rpt.parameter_gaps) if gap_rpt else {}
        param_status_counts = dict(Counter(
            getattr(ext_params, f).status
            for f in ext_params.__class__.model_fields.keys()
        )) if ext_params else {}

        feat  = result.get('feasibility_report')
        bseq  = result.get('build_sequence')
        cg    = result.get('component_graph')
        arpt  = result.get('report')
        meta  = result.get('metadata')
        pdoc  = result.get('paper_doc')

        paper_result = {
            'paper_id'            : paper_id,
            'pdf_name'            : pdf_name,
            'status'              : 'SUCCESS',
            'elapsed_seconds'     : elapsed,
            'title'               : meta.title if meta else 'N/A',
            'authors_count'       : len(meta.authors) if meta else 0,
            'sections_count'      : len(pdoc.sections) if pdoc else 0,
            'tables_count'        : len(pdoc.tables)   if pdoc else 0,
            'components_count'    : len(cg.components) if cg else 0,
            'edges_count'         : len(cg.edges)      if cg else 0,
            'param_status_counts' : param_status_counts,
            'gap_counts'          : dict(gap_counts),
            'has_critical_missing': gap_rpt.has_critical_missing_parameters if gap_rpt else None,
            'feasibility_status'  : feat.overall_status if feat else 'N/A',
            'milestones_count'    : len(bseq.milestones) if bseq else 0,
            'total_duration_weeks': bseq.total_duration_weeks if bseq else 0,
            '_result_full'        : result
        }
        all_results.append(paper_result)

        print(f'  [OK] Done in {elapsed}s')
        print(f'       Title       : {paper_result["title"][:70]}')
        print(f'       Components  : {paper_result["components_count"]}  Edges: {paper_result["edges_count"]}')
        print(f'       Gaps        : {dict(gap_counts)}')
        print(f'       Feasibility : {paper_result["feasibility_status"]}')

    except Exception as e:
        elapsed = round(time.time() - t0, 2)
        print(f'  [ERROR] {pdf_name} failed after {elapsed}s: {e}')
        all_errors.append({
            'paper_id'       : paper_id,
            'pdf_name'       : pdf_name,
            'status'         : 'ERROR',
            'elapsed_seconds': elapsed,
            'error'          : str(e),
            'traceback'      : traceback.format_exc()
        })
        if not SKIP_ON_ERROR:
            raise

total_run_time = round(time.time() - run_start_time, 2)
print()
print('=' * 65)
print(f'RUN COMPLETE: {len(all_results)} success, {len(all_errors)} errors')
print(f'Total time  : {total_run_time}s ({round(total_run_time/60, 1)} min)')
print('=' * 65)


[ 1/29] Processing: [1].pdf


c:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\Projects\agentic_projects\Paper-2-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



[Orchestrator] Step 1: Parsing and merging paper 'c:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\Projects\agentic_projects\Paper-2-Project\backend\papers\research_papers\[1].pdf'...
[2026-08-23 23:44:55] [INFO] [paper_to_project] 🚀 Starting routed extraction pipeline for '[1].pdf'...
[2026-08-23 23:44:55] [INFO] [paper_to_project] Routing '[1].pdf' to PyMuPDF text & section parser.
[2026-08-23 23:44:55] [INFO] [paper_to_project] Extracting blocks from PDF '[1].pdf' using two_column layout...
[2026-08-23 23:45:01] [INFO] [paper_to_project] Detecting sections for paper: 'A Novel Change Detection Method Based on Visual'...
[2026-08-23 23:45:01] [INFO] [paper_to_project] Section pruning triggered by header: 'REFERENCES'
[2026-08-23 23:45:01] [INFO] [paper_to_project] Checking GROBID server availability at http://localhost:8070...
[2026-08-23 23:45:01] [INFO] [paper_to_project] GROBID is active. Routing '[1].pdf' to GROBID.
[2026-08-23 23:45:01] [INFO] [paper_to_project] Sending docu

[INFO] 2026-08-23 23:45:19,949 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-23 23:45:20,015 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\Projects\agentic_projects\Paper-2-Project\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-23 23:45:20,016 [RapidOCR] main.py:63: Using C:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\Projects\agentic_projects\Paper-2-Project\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-23 23:45:20,075 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-23 23:45:20,098 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\Projects\agentic_projects\Paper-2-Project\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-23 23:45:20,099 [RapidOCR] main.py:63: Using C:\Users\kvcsu_ht23nk8\OneDrive\Desktop\al

[2026-08-23 23:46:06] [INFO] [paper_to_project] Docling conversion completed successfully for '[1].pdf'.
[2026-08-23 23:46:06] [INFO] [paper_to_project] [FINISH] Finished routed extraction for '[1].pdf'. Selected: ['pymupdf', 'grobid', 'docling']
[2026-08-23 23:46:06] [INFO] [paper_to_project] Merging extraction outputs for '[1].pdf' into canonical PaperDocument...
[Orchestrator] Slicing document and saving embeddings in pgvector database...
[DB] Database initialized successfully (PostgreSQL + pgvector).
[DB] Saved 97 chunks with vectors for 'paper_1'.
Initializing ChatOllama with model 'qwen2.5-coder:1.5b'...
Sending request to local Ollama for structured metadata extraction...
sections_found.12.character_count
  Field required [type=missing, input_value={'title': 'C. Encoding and'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
primary_contribution
  Field required [type=missing, input_value={'title': 'A Novel Change...e': 'C. Encoding 

...

[2026-08-24 00:18:45] [INFO] [paper_to_project] Docling conversion completed successfully for '[21].pdf'.
[2026-08-24 00:18:45] [INFO] [paper_to_project] [FINISH] Finished routed extraction for '[21].pdf'. Selected: ['pymupdf', 'grobid', 'docling']
[2026-08-24 00:18:45] [INFO] [paper_to_project] Merging extraction outputs for '[21].pdf' into canonical PaperDocument...
[Orchestrator] Slicing document and saving embeddings in pgvector database...
[DB] Database initialized successfully (PostgreSQL + pgvector).
[DB] Saved 71 chunks with vectors for 'paper_21'.
Initializing ChatOllama with model 'qwen2.5-coder:1.5b'...
Sending request to local Ollama for structured metadata extraction...

[Orchestrator] Step 2: Running Method Decomposition Agent...
[Decomposition Agent] Querying local pgvector database for grounded architectural evidence...
[Decomposition Agent] Grounded context loaded (5 evidence packages).
Initializing ChatOllama with model 'qwen2.5-coder:1.5b'...
Sending request to local

[WARNING] 2026-08-24 00:22:42,654 [RapidOCR] main.py:132: The text detection result is empty
RapidOCR returned empty result!
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


[2026-08-24 00:22:46] [INFO] [paper_to_project] Docling conversion completed successfully for '[24].pdf'.
[2026-08-24 00:22:46] [INFO] [paper_to_project] [FINISH] Finished routed extraction for '[24].pdf'. Selected: ['pymupdf', 'grobid', 'docling']
[2026-08-24 00:22:46] [INFO] [paper_to_project] Merging extraction outputs for '[24].pdf' into canonical PaperDocument...
[Orchestrator] Slicing document and saving embeddings in pgvector database...
[DB] Database initialized successfully (PostgreSQL + pgvector).
[DB] Saved 37 chunks with vectors for 'paper_24'.
Initializing ChatOllama with model 'qwen2.5-coder:1.5b'...
Sending request to local Ollama for structured metadata extraction...

[Orchestrator] Step 2: Running Method Decomposition Agent...
[Decomposition Agent] Querying local pgvector database for grounded architectural evidence...
[Decomposition Agent] Grounded context loaded (5 evidence packages).
Initializing ChatOllama with model 'qwen2.5-coder:1.5b'...
Sending request to local

Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


[2026-08-24 00:24:03] [INFO] [paper_to_project] Docling conversion completed successfully for '[25].pdf'.
[2026-08-24 00:24:03] [INFO] [paper_to_project] [FINISH] Finished routed extraction for '[25].pdf'. Selected: ['pymupdf', 'grobid', 'docling']
[2026-08-24 00:24:03] [INFO] [paper_to_project] Merging extraction outputs for '[25].pdf' into canonical PaperDocument...
[Orchestrator] Slicing document and saving embeddings in pgvector database...
[DB] Database initialized successfully (PostgreSQL + pgvector).
[DB] Saved 17 chunks with vectors for 'paper_25'.
Initializing ChatOllama with model 'qwen2.5-coder:1.5b'...
Sending request to local Ollama for structured metadata extraction...

[Orchestrator] Step 2: Running Method Decomposition Agent...
[Decomposition Agent] Querying local pgvector database for grounded architectural evidence...
[Decomposition Agent] Grounded context loaded (5 evidence packages).
Initializing ChatOllama with model 'qwen2.5-coder:1.5b'...
Sending request to local

## Cell 4: Per-Paper Summary Table

In [7]:
header = f"{'#':<4} {'PDF':<12} {'STATUS':<9} {'TIME(s)':<9} {'COMPS':<6} {'EDGES':<6} {'FEASIBILITY':<13} {'EXPLICIT':<10} {'MISSING':<8} TITLE"
print(header)
print('-' * 130)
for i, r in enumerate(all_results, 1):
    gc = r.get('gap_counts', {})
    print(
        f"{i:<4} {r['pdf_name']:<12} {'OK':<9} {r['elapsed_seconds']:<9} "
        f"{r['components_count']:<6} {r['edges_count']:<6} {r['feasibility_status']:<13} "
        f"{gc.get('EXPLICIT',0):<10} {gc.get('MISSING',0):<8} {r['title'][:45]}"
    )
for r in all_errors:
    print(
        f"{'--':<4} {r['pdf_name']:<12} {'ERROR':<9} {r['elapsed_seconds']:<9} "
        f"{'--':<6} {'--':<6} {'N/A':<13} {'--':<10} {'--':<8} {r['error'][:50]}"
    )
print()
print(f'Total: {len(all_results)} success / {len(all_errors)} errors / {len(papers_to_run)} total')

#    PDF          STATUS    TIME(s)   COMPS  EDGES  FEASIBILITY   EXPLICIT   MISSING  TITLE
----------------------------------------------------------------------------------------------------------------------------------


NameError: name 'all_results' is not defined

## Cell 5: Corpus-Wide Aggregate Statistics

In [6]:
if all_results:
    all_feasibility  = Counter(r['feasibility_status']  for r in all_results)
    all_gap_counts   = Counter()
    all_param_status = Counter()
    total_components = sum(r['components_count'] for r in all_results)
    total_edges      = sum(r['edges_count'] for r in all_results)
    avg_elapsed      = round(sum(r['elapsed_seconds'] for r in all_results) / len(all_results), 1)
    for r in all_results:
        all_gap_counts   += Counter(r.get('gap_counts', {}))
        all_param_status += Counter(r.get('param_status_counts', {}))
    critical_missing_count = sum(1 for r in all_results if r.get('has_critical_missing'))

    print('=' * 55)
    print('CORPUS-WIDE AGGREGATE STATISTICS')
    print('=' * 55)
    print(f'  System            : {gpu_model}')
    print(f'  VRAM              : {vram_gb} GB | RAM: {system_ram_gb} GB')
    print(f'  Model             : {MODEL_NAME}')
    print()
    print(f'  Papers processed  : {len(all_results)} / {len(papers_to_run)}')
    print(f'  Papers failed     : {len(all_errors)}')
    print(f'  Avg time / paper  : {avg_elapsed}s')
    print()
    print(f'  Total components  : {total_components}')
    print(f'  Total edges       : {total_edges}')
    print(f'  Avg comps / paper : {round(total_components/len(all_results),1)}')
    print(f'  Avg edges / paper : {round(total_edges/len(all_results),1)}')
    print()
    print('  --- Feasibility Distribution ---')
    for k, v in sorted(all_feasibility.items()):
        print(f'    {k:<15}: {v} paper(s)')
    print()
    print('  --- Gap Classification Distribution ---')
    for k, v in sorted(all_gap_counts.items()):
        print(f'    {k:<15}: {v} parameters')
    print(f'  Critical missing in {critical_missing_count} paper(s)')
    print()
    print('  --- Parameter Status Distribution ---')
    for k, v in sorted(all_param_status.items()):
        print(f'    {k:<15}: {v} parameters')

NameError: name 'all_results' is not defined

## Cell 6: Save Consolidated Report (Markdown + JSON)

In [5]:
report_md_path   = os.path.join(REPORTS_DIR, f'corpus_report_{RUN_TS}.md')
report_json_path = os.path.join(REPORTS_DIR, f'corpus_report_{RUN_TS}.json')

# ==============================================================
#  MARKDOWN REPORT
# ==============================================================
md = []
md.append('# Paper-to-Project: Phase 1–5 Corpus-Wide Report')
md.append('')
md.append(f'**Generated:** {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
md.append(f'**Model:** `{MODEL_NAME}`')
md.append(f'**GPU:** {gpu_model} ({vram_gb} GB VRAM) | **RAM:** {system_ram_gb} GB')
md.append(f'**Papers Run:** {len(papers_to_run)} | **Success:** {len(all_results)} | **Errors:** {len(all_errors)}')
md.append(f'**Total Runtime:** {round(total_run_time/60, 1)} min')
md.append('')
md.append('---')
md.append('')

if all_results:
    md.append('## Corpus Aggregate Statistics')
    md.append('')
    md.append('| Metric | Value |')
    md.append('|--------|-------|')
    md.append(f'| Papers Processed | {len(all_results)} |')
    md.append(f'| Papers Failed | {len(all_errors)} |')
    md.append(f'| Avg Time / Paper | {avg_elapsed}s |')
    md.append(f'| Total Components | {total_components} |')
    md.append(f'| Total Edges | {total_edges} |')
    md.append(f'| Avg Components / Paper | {round(total_components/len(all_results),1)} |')
    md.append(f'| Papers with Critical Missing Gaps | {critical_missing_count} |')
    md.append('')
    md.append('### Feasibility Distribution')
    md.append('| Status | Count |')
    md.append('|--------|-------|')
    for k, v in sorted(all_feasibility.items()):
        md.append(f'| {k} | {v} |')
    md.append('')
    md.append('### Gap Classification Distribution')
    md.append('| Classification | Total |')
    md.append('|----------------|-------|')
    for k, v in sorted(all_gap_counts.items()):
        md.append(f'| {k} | {v} |')
    md.append('')
    md.append('---')
    md.append('')

# Per-paper summary table
md.append('## Per-Paper Results')
md.append('')
md.append('| # | PDF | Status | Time(s) | Comps | Edges | Feasibility | EXPLICIT | MISSING | Title |')
md.append('|---|-----|--------|---------|-------|-------|-------------|----------|---------|-------|')
for i, r in enumerate(all_results, 1):
    gc = r.get('gap_counts', {})
    md.append(
        f"| {i} | {r['pdf_name']} | OK | {r['elapsed_seconds']} | "
        f"{r['components_count']} | {r['edges_count']} | {r['feasibility_status']} | "
        f"{gc.get('EXPLICIT',0)} | {gc.get('MISSING',0)} | {r['title'][:50].replace('|','-')} |"
    )
for r in all_errors:
    md.append(
        f"| - | {r['pdf_name']} | ERROR | {r['elapsed_seconds']} | - | - | - | - | - | "
        f"{r['error'][:50].replace('|','-')} |"
    )
md.append('')
md.append('---')
md.append('')

# Per-paper detailed sections
md.append('## Per-Paper Detailed Reports')
md.append('')
for i, r in enumerate(all_results, 1):
    res  = r['_result_full']
    cg   = res.get('component_graph')
    grpt = res.get('gap_report')
    feat = res.get('feasibility_report')
    bseq = res.get('build_sequence')
    extp = res.get('extracted_parameters')
    arpt = res.get('report')

    md.append(f'### [{i}] {r["pdf_name"]} — {r["title"][:80]}')
    md.append('')
    md.append(f'- **Elapsed:** {r["elapsed_seconds"]}s | **Sections:** {r["sections_count"]} | **Tables:** {r["tables_count"]}')
    md.append(f'- **Components:** {r["components_count"]} | **Edges:** {r["edges_count"]}')
    md.append(f'- **Feasibility:** {r["feasibility_status"]} | **Milestones:** {r["milestones_count"]} ({r["total_duration_weeks"]} weeks)')
    md.append('')

    if extp:
        md.append('**Parameters:**')
        md.append('| Parameter | Value | Status | Confidence |')
        md.append('|-----------|-------|--------|------------|')
        for field_name in extp.__class__.model_fields.keys():
            p = getattr(extp, field_name)
            md.append(f'| {field_name.upper()} | {p.value} | {p.status} | {p.confidence:.2f} |')
        md.append('')

    if grpt and grpt.parameter_gaps:
        md.append('**Gap Classification:**')
        md.append('| Parameter | Classification | Value |')
        md.append('|-----------|---------------|-------|')
        for gap in grpt.parameter_gaps:
            md.append(f'| {gap.parameter_name.upper()} | {gap.classification} | {gap.value} |')
        md.append(f'*Has critical missing: {grpt.has_critical_missing_parameters}*')
        md.append('')

    if bseq and bseq.milestones:
        md.append('**Build Sequence:**')
        md.append('| # | Milestone | Days | Priority |')
        md.append('|---|-----------|------|----------|')
        for j, ms in enumerate(bseq.milestones, 1):
            md.append(f'| {j} | {ms.name[:60]} | {ms.estimated_duration_days} | {ms.priority} |')
        md.append('')

    if arpt:
        md.append('**Executive Summary:**')
        md.append(f'> {arpt.executive_summary[:300]}...')
        md.append('')

    md.append('---')
    md.append('')

if all_errors:
    md.append('## Errors')
    md.append('')
    for err in all_errors:
        md.append(f'### {err["pdf_name"]}')
        md.append('```')
        md.append(err['error'])
        md.append('```')
        md.append('')

with open(report_md_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(md))
print(f'[OK] Markdown report : {report_md_path}')

# ==============================================================
#  JSON REPORT
# ==============================================================
def safe_serialize(r):
    res  = r.get('_result_full', {})
    cg   = res.get('component_graph')
    grpt = res.get('gap_report')
    feat = res.get('feasibility_report')
    bseq = res.get('build_sequence')
    extp = res.get('extracted_parameters')
    pdoc = res.get('paper_doc')
    arpt = res.get('report')
    meta = res.get('metadata')
    return {
        'paper_id'        : r['paper_id'],
        'pdf_name'        : r['pdf_name'],
        'status'          : r['status'],
        'elapsed_seconds' : r['elapsed_seconds'],
        'metadata': {
            'title'   : meta.title    if meta else 'N/A',
            'authors' : meta.authors  if meta else [],
            'abstract': meta.abstract[:500] if meta else '',
            'primary_contribution': meta.primary_contribution if meta else ''
        },
        'paper_doc_stats': {
            'sections' : len(pdoc.sections)  if pdoc else 0,
            'tables'   : len(pdoc.tables)    if pdoc else 0,
            'equations': len(pdoc.equations) if pdoc else 0
        },
        'component_graph': {
            'components': [{'name': c.name, 'type': c.type, 'params': list(c.parameters.keys())} for c in cg.components] if cg else [],
            'edges'     : cg.edges if cg else []
        },
        'extracted_parameters': {
            field: {'value': getattr(extp, field).value, 'status': getattr(extp, field).status, 'confidence': getattr(extp, field).confidence}
            for field in extp.__class__.model_fields.keys()
        } if extp else {},
        'gap_report': {
            'summary'             : grpt.summary if grpt else '',
            'has_critical_missing': grpt.has_critical_missing_parameters if grpt else None,
            'gaps': [{'parameter': g.parameter_name, 'classification': g.classification, 'value': g.value, 'details': g.details} for g in grpt.parameter_gaps] if grpt else []
        },
        'feasibility': {
            'overall_status'     : feat.overall_status      if feat else 'N/A',
            'training_status'    : feat.training_status     if feat else 'N/A',
            'training_substitute': feat.training_substitute if feat else '',
            'components': [{'name': cf.component_name, 'status': cf.status, 'reason': cf.reason} for cf in feat.components] if feat else []
        },
        'build_sequence': {
            'total_duration_weeks': bseq.total_duration_weeks if bseq else 0,
            'milestones': [{'name': ms.name, 'duration_days': ms.estimated_duration_days, 'priority': ms.priority} for ms in bseq.milestones] if bseq else []
        },
        'adaptation_report': {
            'executive_summary'    : arpt.executive_summary     if arpt else '',
            'bottleneck_analysis'  : arpt.bottleneck_analysis   if arpt else '',
            'cloud_migration_guide': arpt.cloud_migration_guide if arpt else ''
        }
    }

json_report = {
    'generated_at'    : datetime.datetime.now().isoformat(),
    'system': {
        'model'         : MODEL_NAME,
        'gpu'           : gpu_model,
        'vram_gb'       : vram_gb,
        'system_ram_gb' : system_ram_gb,
        'timeline_weeks': TIMELINE_WEEKS
    },
    'papers_total'    : len(papers_to_run),
    'papers_success'  : len(all_results),
    'papers_error'    : len(all_errors),
    'total_runtime_sec': total_run_time,
    'aggregate': {
        'total_components'     : total_components if all_results else 0,
        'total_edges'          : total_edges      if all_results else 0,
        'avg_elapsed_seconds'  : avg_elapsed      if all_results else 0,
        'feasibility_dist'     : dict(all_feasibility)  if all_results else {},
        'gap_class_dist'       : dict(all_gap_counts)   if all_results else {},
        'param_status_dist'    : dict(all_param_status) if all_results else {},
        'critical_missing_count': critical_missing_count if all_results else 0
    },
    'papers': [safe_serialize(r) for r in all_results],
    'errors': all_errors
}

with open(report_json_path, 'w', encoding='utf-8') as f:
    json.dump(json_report, f, indent=2, ensure_ascii=False)
print(f'[OK] JSON report     : {report_json_path}')

print()
print('============================')
print('ALL REPORTS SAVED')
print('============================')

NameError: name 'RUN_TS' is not defined

## Cell 7: Final Scorecard

In [ ]:
print('=' * 65)
print('PHASE 1-5 CORPUS SCORECARD')
print('=' * 65)
print(f'  GPU              : {gpu_model}')
print(f'  VRAM             : {vram_gb} GB | RAM: {system_ram_gb} GB')
print(f'  Model            : {MODEL_NAME}')
print()
print(f'  Papers total     : {len(papers_to_run)}')
print(f'  Success          : {len(all_results)}')
print(f'  Errors           : {len(all_errors)}')
if all_errors:
    print(f'  Failed           : {", ".join(e["pdf_name"] for e in all_errors)}')
print()
if all_results:
    print(f'  Avg time/paper   : {avg_elapsed}s')
    print(f'  Total components : {total_components}')
    print(f'  Total edges      : {total_edges}')
    print()
    print(f'  --- Feasibility ---')
    for k, v in sorted(all_feasibility.items()):
        pct = round(v / len(all_results) * 100)
        print(f'    {k:<15}: {v:>3} ({pct}%)')
    print()
    print(f'  --- Gap Classifications ---')
    for k, v in sorted(all_gap_counts.items()):
        print(f'    {k:<15}: {v:>4} parameters')
    print(f'  Critical missing in {critical_missing_count}/{len(all_results)} papers')
print()
print(f'  Reports saved to : {REPORTS_DIR}')
print(f'    MD   : corpus_report_{RUN_TS}.md')
print(f'    JSON : corpus_report_{RUN_TS}.json')
print('=' * 65)
print('DONE')
print('=' * 65)